In [1]:
import os
import glob
import logging
from pathlib import Path
import zarr

import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

In [2]:
ERA5_INST_GLOB  = "data/raw/ca_2023_2026_era5_hourly/era5_inst_*.nc"
ERA5_ACCUM_GLOB = "data/raw/ca_2023_2026_era5_hourly/era5_accum_*.nc"
LMP_WIDE_PATH   = "data/processed/caiso_lmp_wide_with_load_and_fuel.parquet"
OUTPUT_DIR      = Path("data/aligned")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# CAISO zone bounding boxes [N, S, W, E]
ZONE_BOXES = {
    "NP15": {"lat": (36.5, 42.0), "lon": (-124.5, -119.5)},
    "SP15": {"lat": (32.5, 36.5), "lon": (-120.5, -114.5)},
    "ZP26": {"lat": (35.0, 37.5), "lon": (-121.0, -118.0)},
}

# GRIB short name → clean name mapping
VAR_RENAME = {
    "t2m":  "t2m",
    "d2m":  "d2m",
    "u10":  "u10",
    "v10":  "v10",
    "sp":   "sp",
    "tcc":  "tcc",
    "tcwv": "tcwv",
    "ssrd": "ssrd",
    "fdir": "fdir",
    "tp":   "tp",
}

INSTANT_VARS  = ["t2m", "d2m", "u10", "v10", "sp", "tcc", "tcwv"]
ACCUM_VARS    = ["ssrd", "fdir", "tp"]
ALL_VARS      = INSTANT_VARS + ACCUM_VARS

In [3]:
# ── LOAD ERA5 ─────────────────────────────────────────────────────────────────

def load_era5(inst_glob: str, accum_glob: str) -> xr.Dataset:
    """
    Load and merge all monthly inst + accum NetCDF files.
    Returns a single Dataset with dim 'valid_time' in UTC (tz-naive).
    """
    inst_files  = sorted(glob.glob(inst_glob))
    accum_files = sorted(glob.glob(accum_glob))

    if not inst_files:
        raise FileNotFoundError(f"No files matched: {inst_glob}")
    if not accum_files:
        raise FileNotFoundError(f"No files matched: {accum_glob}")

    log.info(f"Loading {len(inst_files)} inst files, {len(accum_files)} accum files...")

    ds_inst  = xr.open_mfdataset(inst_files,  combine="by_coords", engine="netcdf4")
    ds_accum = xr.open_mfdataset(accum_files, combine="by_coords", engine="netcdf4")

    # Merge on shared time + lat/lon
    ds = xr.merge([ds_inst, ds_accum], compat="override")

    # Standardize time dim name (cfgrib uses 'valid_time', plain netcdf uses 'time')
    if "valid_time" in ds.dims and "time" not in ds.dims:
        ds = ds.rename({"valid_time": "time"})

    # Drop scalar coords that cause issues downstream
    scalar_coords = [c for c in ds.coords if ds[c].dims == ()]
    ds = ds.drop_vars(scalar_coords, errors="ignore")

    log.info(f"ERA5 loaded: {len(ds.time)} hours, "
             f"lat {float(ds.latitude.min()):.2f}–{float(ds.latitude.max()):.2f}, "
             f"lon {float(ds.longitude.min()):.2f}–{float(ds.longitude.max()):.2f}")
    return ds


In [4]:
# ── SPATIAL CROP PER ZONE ─────────────────────────────────────────────────────

def crop_zone(ds: xr.Dataset, zone: str) -> xr.Dataset:
    """
    Crop ERA5 to zone bounding box.
    ERA5 latitude decreases north→south, longitude increases west→east.
    """
    box = ZONE_BOXES[zone]
    lat_lo, lat_hi = box["lat"]
    lon_lo, lon_hi = box["lon"]

    # ERA5 lat dim is descending → use slice(hi, lo)
    ds_zone = ds.sel(
        latitude=slice(lat_hi, lat_lo),
        longitude=slice(lon_lo, lon_hi),
    )

    n_cells = len(ds_zone.latitude) * len(ds_zone.longitude)
    log.info(f"  {zone}: {len(ds_zone.latitude)} lat × {len(ds_zone.longitude)} lon = {n_cells} cells")
    return ds_zone


In [5]:
def make_zone_averaged(ds: xr.Dataset) -> pd.DataFrame:
    """
    Spatial average over each zone box → one scalar per (variable, zone, hour).
    Returns wide DataFrame: one row per hour, columns = variable_ZONE.

    e.g. t2m_NP15, ssrd_SP15, u10_ZP26, ...
    """
    zone_dfs = []

    for zone in ZONE_BOXES:
        log.info(f"Averaging zone {zone}...")
        ds_zone = crop_zone(ds, zone)

        # Spatial mean over lat/lon → (time,) per variable
        zone_mean = ds_zone.mean(dim=["latitude", "longitude"])

        df = zone_mean.to_dataframe()[ALL_VARS].copy()
        df.index.name = "time"
        df.reset_index(inplace=True)

        # Rename: t2m → t2m_NP15
        df.rename(columns={v: f"{v}_{zone}" for v in ALL_VARS}, inplace=True)
        zone_dfs.append(df)

    # Merge all zones on time
    era5_wide = zone_dfs[0]
    for df in zone_dfs[1:]:
        era5_wide = era5_wide.merge(df, on="time", how="inner")

    # Add derived features useful for LMP
    for zone in ZONE_BOXES:
        t   = era5_wide[f"t2m_{zone}"]
        td  = era5_wide[f"d2m_{zone}"]
        # Relative humidity proxy (Magnus approximation)
        era5_wide[f"rh_{zone}"] = 100 * np.exp(
            17.625 * (td - 273.15) / (243.04 + td - 273.15) -
            17.625 * (t  - 273.15) / (243.04 + t  - 273.15)
        )
        # Wind speed scalar
        era5_wide[f"wind_speed_{zone}"] = np.sqrt(
            era5_wide[f"u10_{zone}"]**2 + era5_wide[f"v10_{zone}"]**2
        )

    log.info(f"Zone-averaged ERA5: {len(era5_wide)} rows, {len(era5_wide.columns)} columns")
    return era5_wide

In [15]:
def make_spatial_zarr(ds: xr.Dataset) -> dict[str, Path]:
    """
    Save one zarr store per zone with full spatial grid preserved.
    Shape: (time, lat, lon, variables) via DataArray stack.

    These are the inputs for the spatiotemporal transformer's spatial encoder.
    """
    output_paths = {}

    for zone in ZONE_BOXES:
        out_path = OUTPUT_DIR / f"era5_spatial_{zone}.zarr"

        if out_path.exists():
            log.info(f"  Zarr cache hit: {out_path.name}")
            output_paths[zone] = out_path
            continue

        log.info(f"Writing spatial zarr for {zone}...")
        ds_zone = crop_zone(ds, zone)

        # Keep only the variables we need
        ds_zone = ds_zone[ALL_VARS]

        # Rechunk for efficient time-slicing (model will iterate over time windows)
        ds_zone = ds_zone.chunk({
            "time": 168,          # one lookback window per chunk
            "latitude": -1,       # full spatial dim in memory
            "longitude": -1,
        })

        ds_zone.to_zarr(out_path, mode="w")
        log.info(f"  Saved: {out_path} "
                 f"({ds_zone.dims['latitude']}×{ds_zone.dims['longitude']} cells)")
        output_paths[zone] = out_path

    return output_paths

In [6]:
def align_with_lmp(era5_wide: pd.DataFrame, lmp_path: str) -> pd.DataFrame:
    """
    Timezone-convert ERA5 UTC → US/Pacific, then inner-join with LMP wide parquet.
    """
    # ERA5 time is UTC tz-naive → localize then convert
    era5_wide["time"] = (
        pd.to_datetime(era5_wide["time"], utc=True)
          .dt.tz_convert("US/Pacific")
    )

    # Load LMP
    lmp = pd.read_parquet(lmp_path)
    lmp["time"] = pd.to_datetime(lmp["time"])
    if lmp["time"].dt.tz is None:
        lmp["time"] = lmp["time"].dt.tz_localize("US/Pacific")

    n_before = len(era5_wide)
    merged = lmp.merge(era5_wide, on="time", how="inner")
    n_after = len(merged)

    log.info(f"LMP rows: {len(lmp):,}  ERA5 rows: {n_before:,}  "
             f"Merged: {n_after:,} ({n_after/len(lmp)*100:.1f}% overlap)")

    if n_after < 0.8 * len(lmp):
        log.warning("⚠️  <80% overlap — check timezone alignment or date range mismatch")

    return merged

In [7]:
# ── Load ERA5 ────────────────────────────────────────────────────────────
ds = load_era5(ERA5_INST_GLOB, ERA5_ACCUM_GLOB)

2026-04-22 16:13:09,777 INFO Loading 37 inst files, 37 accum files...
2026-04-22 16:13:12,714 INFO ERA5 loaded: 27048 hours, lat 31.75–42.00, lon -124.50–-114.00


In [8]:
# ── Version 1: Zone-averaged parquet ─────────────────────────────────────
log.info("\n── Version 1: Zone-averaged (iTransformer baseline) ──")
era5_wide   = make_zone_averaged(ds)
merged      = align_with_lmp(era5_wide, LMP_WIDE_PATH)

2026-04-22 16:13:20,296 INFO 
── Version 1: Zone-averaged (iTransformer baseline) ──
2026-04-22 16:13:20,297 INFO Averaging zone NP15...
2026-04-22 16:13:20,301 INFO   NP15: 23 lat × 21 lon = 483 cells
2026-04-22 16:13:26,590 INFO Averaging zone SP15...
2026-04-22 16:13:26,602 INFO   SP15: 17 lat × 25 lon = 425 cells
2026-04-22 16:13:27,002 INFO Averaging zone ZP26...
2026-04-22 16:13:27,005 INFO   ZP26: 11 lat × 13 lon = 143 cells
2026-04-22 16:13:27,329 INFO Zone-averaged ERA5: 27048 rows, 37 columns
2026-04-22 16:13:27,442 INFO LMP rows: 22,248  ERA5 rows: 27,048  Merged: 22,248 (100.0% overlap)


In [ ]:
log.info("\n── Version 2: Spatial grid zarr (spatiotemporal arch) ──")
zarr_paths = make_spatial_zarr(ds)
for zone, path in zarr_paths.items():
    z = xr.open_zarr(path)
    log.info(f"  {zone}: {dict(z.dims)} — variables: {list(z.data_vars)}")

In [10]:
merged.columns

Index(['time', 'Congestion_NP15', 'Congestion_SP15', 'Congestion_ZP26',
       'Energy_NP15', 'Energy_SP15', 'Energy_ZP26', 'Lmp_NP15', 'Lmp_SP15',
       'Lmp_ZP26', 'Load_NP15', 'Load_SP15', 'Load_ZP26', 'Loss_NP15',
       'Loss_SP15', 'Loss_ZP26', 'Solar_NP15', 'Solar_SP15', 'Solar_ZP26',
       'Wind_NP15', 'Wind_SP15', 'Wind_ZP26', 'Congestion_Spread_SP15_NP15',
       'hour_of_day', 'day_of_week', 'month', 'year', 'is_weekend', 'season',
       't2m_NP15', 'd2m_NP15', 'u10_NP15', 'v10_NP15', 'sp_NP15', 'tcc_NP15',
       'tcwv_NP15', 'ssrd_NP15', 'fdir_NP15', 'tp_NP15', 't2m_SP15',
       'd2m_SP15', 'u10_SP15', 'v10_SP15', 'sp_SP15', 'tcc_SP15', 'tcwv_SP15',
       'ssrd_SP15', 'fdir_SP15', 'tp_SP15', 't2m_ZP26', 'd2m_ZP26', 'u10_ZP26',
       'v10_ZP26', 'sp_ZP26', 'tcc_ZP26', 'tcwv_ZP26', 'ssrd_ZP26',
       'fdir_ZP26', 'tp_ZP26', 'rh_NP15', 'wind_speed_NP15', 'rh_SP15',
       'wind_speed_SP15', 'rh_ZP26', 'wind_speed_ZP26'],
      dtype='object')

In [11]:
avg_path = OUTPUT_DIR / "caiso_lmp_era5_averaged.parquet"
merged.to_parquet(avg_path, index=False)
log.info(f"✓ Saved: {avg_path}")
log.info(f"  Shape: {merged.shape}")
log.info(f"  Columns: {[c for c in merged.columns if 't2m' in c or 'Cong' in c]}")

2026-04-22 16:13:50,063 INFO ✓ Saved: data/aligned/caiso_lmp_era5_averaged.parquet
2026-04-22 16:13:50,064 INFO   Shape: (22248, 65)
2026-04-22 16:13:50,064 INFO   Columns: ['Congestion_NP15', 'Congestion_SP15', 'Congestion_ZP26', 'Congestion_Spread_SP15_NP15', 't2m_NP15', 't2m_SP15', 't2m_ZP26']
